# HEA Hardness KAN — Edge Ablation Study

This notebook investigates how prediction accuracy degrades as edges are
progressively removed from a trained KAN.  The goal is to find the
**minimum edge count that retains acceptable accuracy** — a useful guide
for producing sparse, interpretable models.

## Approach

1. Train a full KAN (hidden=20, 220 edges) using best hyperparameters from the Optuna sweep.
2. Copy the trained model, then apply a three-phase pruning schedule:
   - **Coarse** — remove 20 edges at a time × 8 steps (bulk compression)
   - **Intermediate** — remove 10 edges × 3 steps
   - **Fine** — remove 5 edges × 3 steps
   After each removal the model is retrained with Adam (mini-batch, with
   best-val-loss checkpointing) to allow the surviving edges to compensate.
3. Repeat across **10 random train/val/test splits** to separate pruning
   sensitivity from data-split variance.
4. Plot mean test MAE ± standard error vs edge count.

The full schedule removes 205 of the original 220 edges, leaving a minimum
of ~15 edges at the end of the curve.

## Why random seeds rather than k-fold?

Each seed requires training one full model and then running ~14 prune→retrain
cycles on it — the ablation needs direct access to the model instance.
`k_fold_val` trains models internally and only returns the aggregate MAE,
so random splits are used here instead.  10 seeds gives the same number of
evaluations as 5-fold × 2 models.

In [ ]:
import json
import sys
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from kan import *

sys.path.insert(0, os.path.abspath(''))
from HEA_pyKAN import (
    train_val_test_split,
    best_loss_KAN,
    ablate_edges,
    DEFAULT_ABLATION_SCHEDULE,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 1  Load and normalise data

In [ ]:
DATA_PATH = r"H:\My Drive\Documents and Data\Data\BerryML"
DATA_FILE = "Gorsse2018ML_Data_RawHV.csv"

df   = pd.read_csv(os.path.join(DATA_PATH, DATA_FILE))
cols = list(df.columns)
df   = df[df["HV"].notna()][cols[25:]]

mean    = df.mean()
stdev   = df.std()
norm_df = (df - mean) / stdev

x_data = torch.tensor(norm_df[cols[26:]].values, dtype=torch.float32).to(device)
y_data = torch.tensor(norm_df[cols[25]].values,  dtype=torch.float32).unsqueeze(-1).to(device)

data_norm = stdev["HV"]
print(f"x: {x_data.shape},  y: {y_data.shape}  |  HV std = {data_norm:.1f}")

## 2  Load hyperparameters

We use the best parameters found by the Optuna sweep for `hidden=20`.
The `lamb_entropy` conversion (absolute → model parameter) is applied here.

In [ ]:
HIDDEN = 20

with open(os.path.join(os.path.abspath(''), "best_hyperparams_by_hidden_nodes.json")) as f:
    best_params = json.load(f)

params     = best_params[str(HIDDEN)]["params"]
k          = params["k"]
g          = params["g"]
lamb       = params["lamb"]
model_seed = params["model_seed"]
lamb_entropy = params["lamb_entropy"] / lamb   # convert absolute → model parameter

print(f"hidden={HIDDEN}  k={k}  g={g}  model_seed={model_seed}")
print(f"lamb={lamb:.3e}  lamb_entropy (model)={lamb_entropy:.3e}")

## 3  Ablation schedule

The default schedule in `HEA_pyKAN` was developed empirically on this dataset.
Print it here for transparency — edit the list below if you want to experiment
with different schedules without changing the library.

In [ ]:
print("Default ablation schedule:")
print(f"  {'Phase':<14} {'n_remove':>8}  {'repeats':>7}  {'epochs':>6}  {'batch':>5}  {'lr':>6}")
print("  " + "-" * 52)
phase_names = ["Coarse", "Intermediate", "Fine"]
for name, phase in zip(phase_names, DEFAULT_ABLATION_SCHEDULE):
    print(f"  {name:<14} {phase['n_remove']:>8}  {phase['repeats']:>7}  "
          f"{phase['epochs']:>6}  {phase['batch_size']:>5}  {phase['lr']:>6}")

total_removed = sum(p["n_remove"] * p["repeats"] for p in DEFAULT_ABLATION_SCHEDULE)
n_initial     = 11 * HIDDEN   # 10 inputs × hidden + hidden × 1 output
print(f"\nTotal edges removed: {total_removed}  "
      f"({n_initial} initial → {n_initial - total_removed} remaining)")

# uncomment to customise:
# schedule = [
#     {"n_remove": 20, "epochs": 20, "batch_size": 128, "lr": 2e-3, "repeats": 8},
#     {"n_remove": 10, "epochs": 20, "batch_size": 32,  "lr": 2e-3, "repeats": 3},
#     {"n_remove":  5, "epochs": 20, "batch_size": 32,  "lr": 2e-3, "repeats": 3},
# ]
schedule = DEFAULT_ABLATION_SCHEDULE

## 4  Ablation loop across dataset seeds

For each seed:
1. Split the data into train (80%) / val (10%) / test (10%) using `train_val_test_split`.
2. Train a fresh KAN with `best_loss_KAN.best_loss_fit` (LBFGS, 25 steps,
   best-val-loss checkpointing).
3. Copy the trained model so the original is preserved, then run `ablate_edges`
   which applies the schedule and returns `(edge_counts, maes)` at each checkpoint.

The model is copied before ablation so that the full 220-edge model is always
the starting point for each seed, independent of what happened in previous seeds.

In [ ]:
N_SEEDS   = 10
all_edges = []   # list of edge-count lists, one per seed
all_maes  = []   # list of MAE lists, one per seed

for seed in range(N_SEEDS):
    print(f"\n{'='*50}")
    print(f"Dataset seed {seed}")
    print(f"{'='*50}")

    # ── split ────────────────────────────────────────────────────────────────
    dataset = train_val_test_split(
        x_data, y_data,
        val_split=0.10, test_split=0.10,
        device=str(device), seed=seed,
    )

    # ── train full model ─────────────────────────────────────────────────────
    model = best_loss_KAN(
        width=[10, HIDDEN, 1], grid=g, k=k,
        seed=model_seed, device=device, auto_save=False,
    ).to(device)

    model.best_loss_fit(
        dataset, opt="LBFGS", steps=25,
        loss_fn=nn.L1Loss(), lamb=lamb, lamb_entropy=lamb_entropy,
        verbose=False,
    )

    # ── copy then ablate ─────────────────────────────────────────────────────
    # copy() so the original trained model is untouched
    abl_model = model.copy()

    edge_counts, maes = ablate_edges(
        abl_model, dataset, data_norm=data_norm,
        schedule=schedule,
        loss_fn=nn.L1Loss,
        optimiser_fn=optim.Adam,
        verbose=True,
    )

    all_edges.append(edge_counts)
    all_maes.append(maes)

print("\nDone.")

## 5  Aggregate results and plot

All seeds follow the same pruning schedule so they produce the same sequence
of edge counts.  We take the edge counts from seed 0 as the x-axis, then
compute mean ± standard error of the MAE across seeds at each checkpoint.

The x-axis is inverted so the curve reads left-to-right as compression
progresses (fewer edges = more pruned).

In [ ]:
edge_axis = np.array(all_edges[0])          # same for every seed
mae_array = np.array(all_maes)              # shape [N_SEEDS, n_checkpoints]

mean_mae   = mae_array.mean(axis=0)
stderr_mae = mae_array.std(axis=0, ddof=1) / np.sqrt(N_SEEDS)

fig, ax = plt.subplots(figsize=(7, 4))

ax.errorbar(
    edge_axis, mean_mae, yerr=stderr_mae,
    marker="o", capsize=4, color="steelblue", label="mean ± SE",
)

# shade the ±1 SE band for readability
ax.fill_between(
    edge_axis,
    mean_mae - stderr_mae,
    mean_mae + stderr_mae,
    alpha=0.2, color="steelblue",
)

ax.invert_xaxis()
ax.set_xlabel("Number of edges")
ax.set_ylabel("Test MAE (HV)")
ax.set_title(f"KAN edge ablation  |  hidden={HIDDEN}  |  {N_SEEDS} seeds")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFull model ({edge_axis[0]} edges): {mean_mae[0]:.1f} ± {stderr_mae[0]:.1f} HV")
print(f"Most pruned ({edge_axis[-1]} edges): {mean_mae[-1]:.1f} ± {stderr_mae[-1]:.1f} HV")
print(f"\nCheckpoint summary:")
print(f"  {'edges':>6}  {'mean MAE':>9}  {'SE':>6}")
for e, m, s in zip(edge_axis, mean_mae, stderr_mae):
    print(f"  {e:>6}  {m:>9.2f}  {s:>6.2f}")

## 6  Visualise pruned architectures — one continuous ablation run

Rather than retraining from scratch for each target edge count, we run a
**single continuous ablation** and take snapshots at chosen waypoints.  This
lets you watch the network architecture evolve as it is progressively
compressed, which is more informative than comparing independently trained
models.

**Strategy:**
1. Train one full model on a fixed seed.
2. Ablate to the first (least pruned) target using the default schedule —
   `ablate_edges` stops as soon as `n_edge ≤ target`.
3. Copy the model, apply `.prune()` / `.prune_input()` to collapse disconnected
   structure, evaluate, and plot.
4. Continue ablating the *same* model from where it stopped, using a fine
   schedule (small steps) to reach the next target.
5. Repeat until all targets are visualised.

The fine connecting schedule between waypoints uses small removals so the
model has a chance to recover after each step before the next snapshot.

In [ ]:
# ── configuration ────────────────────────────────────────────────────────────
# Waypoints to snapshot at. Must be in descending order (most → least edges).
# Pick values that sit at interesting points on the ablation curve above.
TARGET_EDGE_COUNTS = [25, 20, 15]

# Fine connecting schedule used between waypoints.
# Small removals + retraining give the surviving edges time to adapt.
FINE_SCHEDULE = [
    {"n_remove": 5, "epochs": 20, "batch_size": 32, "lr": 2e-3, "repeats": 20},
]

VIS_SEED = 0   # fixed so architecture differences reflect pruning, not split luck

# ── train one full model ──────────────────────────────────────────────────────
print("Training full model...")
dataset_vis = train_val_test_split(
    x_data, y_data,
    val_split=0.10, test_split=0.10,
    device=str(device), seed=VIS_SEED,
)

model = best_loss_KAN(
    width=[10, HIDDEN, 1], grid=g, k=k,
    seed=model_seed, device=device, auto_save=False,
).to(device)

model.best_loss_fit(
    dataset_vis, opt="LBFGS", steps=25,
    loss_fn=nn.L1Loss(), lamb=lamb, lamb_entropy=lamb_entropy,
    verbose=False,
)

# single working copy — ablated progressively through all targets
abl_model = model.copy()

print(f"Full model: {abl_model.n_edge} edges")

# ── continuous ablation with snapshots ───────────────────────────────────────
for i, target in enumerate(TARGET_EDGE_COUNTS):

    # first target: use the default bulk schedule to compress quickly
    # subsequent targets: use the fine schedule to step down carefully
    sched = DEFAULT_ABLATION_SCHEDULE if i == 0 else FINE_SCHEDULE

    print(f"\n{'='*55}")
    print(f"Ablating to ~{target} edges  "
          f"({'default schedule' if i == 0 else 'fine schedule'})")
    print(f"{'='*55}")

    ablate_edges(
        abl_model, dataset_vis, data_norm=data_norm,
        schedule=sched,
        target_edges=target,
        loss_fn=nn.L1Loss,
        optimiser_fn=optim.Adam,
        verbose=True,
    )

    # ── snapshot: copy and clean up disconnected structure ────────────────────
    pruned = abl_model.copy()
    pruned = pruned.prune(edge_th=0, node_th=0)
    pruned = pruned.prune_input(threshold=0)

    with torch.no_grad():
        mae_abl = nn.L1Loss()(
            abl_model(dataset_vis["test_input"]).squeeze(),
            dataset_vis["test_label"].squeeze(),
        ).item() * data_norm

        mae_pruned = nn.L1Loss()(
            pruned(dataset_vis["test_input"]).squeeze(),
            dataset_vis["test_label"].squeeze(),
        ).item() * data_norm

    print(f"\nSnapshot at ~{target} edges:")
    print(f"  Ablated model  : {abl_model.n_edge} edges  MAE = {mae_abl:.1f} HV")
    print(f"  After .prune() : {pruned.n_edge} edges  MAE = {mae_pruned:.1f} HV")
    print(f"  (MAE shift from removing disconnected structure: "
          f"{mae_pruned - mae_abl:+.2f} HV)")

    pruned.plot(scale=2)